# Creating a Chirp Replica

Creating a Chirp and save it to a product folder.

In [ ]:
from typing import Literal

import numpy as np
import numpy.typing as npt

Defining a function to compute a 1D chirp signal fully customizable by the user with the following parameters:

- samples (length)
- sampling frequency
- start frequency
- start phase
- bandwidth
- pulse length
- direction (UP/DOWN)
- delay

In [ ]:
def generate_chirp(
    samples: int,
    sampling_frequency: float,
    start_frequency: float,
    bandwidth: float,
    pulse_length: float,
    direction: Literal["UP", "DOWN"],
    start_phase: float | None = None,
    delay: float | None = None,
) -> npt.NDArray[np.complex64]:
    """Generate a 1D chirp signal.

    Parameters
    ----------
    samples : int
        Length of the generated array.
    sampling_frequency : float
        Sampling frequency [Hz].
    start_frequency : float
        Chirp starting frequency [Hz].
    bandwidth : float
        Chirp bandwidth [Hz].
    pulse_length : float
        Pulse length [s].
    direction : Literal["UP", "DOWN"]
        Chirp direction.
    start_phase : float | None, optional
        Chirp starting phase [rad], if None it's set to 0.0, by default None
    delay : float | None. optional
        Time-domain delay [s], implemented through a linear phase in frequency,
        if None, it's set to 0.0, by default None

    Returns
    -------
    npt.NDArray[np.complex64]
        Chirp array of shape (samples,).
    """
    delay = delay or 0.0
    start_phase = start_phase or 0.0

    axis = np.arange(samples) / sampling_frequency

    direction_factor = 1 if direction == "UP" else -1
    chirp_rate = bandwidth / pulse_length

    # Generate the un-shifted chirp as the starting (reference) signal.
    chirp_axis = axis[axis <= pulse_length]
    chirp_phase_quadratic = np.pi * chirp_rate * chirp_axis**2
    chirp_phase_linear = 2 * np.pi * start_frequency * chirp_axis
    chirp_phase = direction_factor * chirp_phase_quadratic + chirp_phase_linear + start_phase

    chirp_signal = np.zeros(samples, dtype=np.complex64)
    chirp_signal[axis <= pulse_length] = np.exp(1j * chirp_phase)

    if delay:
        # Frequency-domain time shift: X(f) * exp(-j * 2π * f * delay).
        chirp_fft = np.fft.fft(chirp_signal)
        frequencies = np.fft.fftfreq(samples, d=1 / sampling_frequency)
        chirp_fft *= np.exp(-1j * 2 * np.pi * frequencies * delay)
        chirp_signal = np.fft.ifft(chirp_fft).astype(np.complex64)

    return chirp_signal

Defining input parameters for the chirp generation.

In [ ]:
bandwidth = 60e6  # Hz
pulse_length = 50.0e-06  # s
direction = "UP"
start_frequency = bandwidth / 2 if direction == "DOWN" else -bandwidth / 2  # Hz
sampling_rate = 1.1 * bandwidth  # Hz
delay = 10.0e-06  # s
samples = int((1.4 * pulse_length + delay) * sampling_rate)

In [ ]:
chirp_data = generate_chirp(
    samples=samples,
    sampling_frequency=sampling_rate,
    start_frequency=start_frequency,
    bandwidth=bandwidth,
    pulse_length=pulse_length,
    direction=direction,
    delay=delay,
)

Writing chirp to a new product folder.

In [ ]:
from pathlib import Path

from perseo_core.timing import PreciseDateTime

from aresys_io.product import (
    Pulse,
    RasterInfo,
    SwathInfo,
    create_new_metadata,
    create_product_folder,
    is_valid_product_folder,
    write_metadata,
    write_raster_with_raster_info,
)

In [ ]:
replica_path = Path("ChirpReplica")

chirp_pf = create_product_folder(pf_path=replica_path, overwrite_ok=True)

channel_id = 1
channel_metadata_file = chirp_pf.channel_metadata_path(channel_id)
channel_raster_file = chirp_pf.channel_data_path(channel_id)

metadata = create_new_metadata()

raster_info = RasterInfo(
    lines=1,
    lines_start=0,
    lines_start_unit="s",
    lines_step=0,
    lines_step_unit="s",
    samples=samples,
    samples_start=0,
    samples_start_unit="s",
    samples_step=1 / sampling_rate,
    samples_step_unit="s",
    cell_type="FLOAT_COMPLEX",
    file_name=channel_raster_file.name,
    byte_order="LITTLEENDIAN",
)

swath = "S1"
polarization = "VV"
acquisition_start_time = PreciseDateTime.from_numeric_datetime(2026, 1, 1)
swath_info = SwathInfo(
    swath=swath,
    polarization=polarization,
    acquisition_start_time=acquisition_start_time,
)

pulse_info = Pulse(
    pulse_length=pulse_length,
    bandwidth=bandwidth,
    pulse_sampling_rate=sampling_rate,
    pulse_energy=1,
    pulse_start_frequency=start_frequency,
    pulse_start_phase=0,
    pulse_direction=direction,
)

metadata.insert_element(raster_info)
metadata.insert_element(swath_info)
metadata.insert_element(pulse_info)

print(metadata)

write_metadata(metadata_obj=metadata, metadata_file=channel_metadata_file)
write_raster_with_raster_info(
    data=chirp_data.reshape(1, -1),
    raster_file=channel_raster_file,
    raster_info=raster_info,
)

print("\nCheck output PF validity:", is_valid_product_folder(replica_path))